# Fine-tuning Laya para FTTH

Este notebook ejecuta fine-tuning del modelo Laya (DeBERTa-v3) en el dataset FTTH usando GPU de Colab (T4).

**Tiempo estimado:** 30-45 minutos
**GPU:** T4 (16GB VRAM)

In [ ]:
# 1. Instalar dependencias
!pip install peft transformers accelerate torch sentencepiece tiktoken protobuf -q

In [ ]:
# 2. Montar Google Drive (para guardar el modelo)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Descargar el dataset desde GitHub
!git clone https://github.com/tecnodespegue/FTTH-Copilot.git /content/FTTH-Copilot
!cd /content/FTTH-Copilot && git lfs pull

In [ ]:
# 4. Verificar GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
    device = 'cuda'
else:
    device = 'cpu'

In [ ]:
# 5. Imports y configuración
import json
import torch
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

# Constantes
LAYA_MODEL = "microsoft/deberta-v3-base"
FTTH_CLASSES = [
    "NORMAL", "OPTICAL_DEGRADATION", "OPTICAL_FAULT",
    "POWER_FAULT", "DEVICE_FAULT", "UPLINK_FAULT",
    "CONGESTION", "MASS_OUTAGE", "UNKNOWN",
]
CLASS_TO_ID = {c: i for i, c in enumerate(FTTH_CLASSES)}
ID_TO_CLASS = {i: c for c, i in CLASS_TO_ID.items()}

DATASET_PATH = "/content/FTTH-Copilot/dataset/laya-ftth/v3/train/train/laya-format.jsonl"
VAL_PATH = "/content/FTTH-Copilot/dataset/laya-ftth/v3/train/val/laya-format.jsonl"
OUTPUT_DIR = "/content/drive/MyDrive/ftth-laya-v1"

In [ ]:
# 6. Dataset class
class FTTHFineTuningDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.examples = self._load(file_path)
    
    def _load(self, file_path):
        examples = []
        with open(file_path) as f:
            for line in f:
                data = json.loads(line)
                ec = data["answers"].get("event_class", {}).get("choice", "UNKNOWN")
                if ec in CLASS_TO_ID:
                    examples.append({"text": data["input"], "label": CLASS_TO_ID[ec]})
        return examples
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(ex["text"], max_length=self.max_length, 
                            padding="max_length", truncation=True, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels": torch.tensor(ex["label"], dtype=torch.long),
        }

In [ ]:
# 7. Cargar modelo y tokenizer
print("📦 Cargando tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(LAYA_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("📦 Cargando modelo...")
model = AutoModelForSequenceClassification.from_pretrained(
    LAYA_MODEL, num_labels=len(FTTH_CLASSES), ignore_mismatched_sizes=True
)

# Aplicar LoRA
print("⚙️ Configurando LoRA...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query_proj", "key_proj", "value_proj"],
    task_type=TaskType.SEQ_CLS,
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"   Parámetros entrenables: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")

In [ ]:
# 8. Cargar datasets
print("📂 Cargando datasets...")
train_dataset = FTTHFineTuningDataset(DATASET_PATH, tokenizer)
print(f"   Train: {len(train_dataset)} ejemplos")

val_dataset = FTTHFineTuningDataset(VAL_PATH, tokenizer)
print(f"   Val: {len(val_dataset)} ejemplos")

In [ ]:
# 9. Training Arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,  # Usar fp16 en GPU
    report_to="none",
    seed=42,
)

In [ ]:
# 10. Entrenar
print("🚀 Iniciando fine-tuning...")
print(f"   Épocas: {training_args.num_train_epochs}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   Learning rate: {training_args.learning_rate}")
print()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)

trainer.train()

In [ ]:
# 11. Guardar modelo
print(f"\n💾 Guardando modelo en: {OUTPUT_DIR}")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Guardar config
config = {
    "model_name": LAYA_MODEL,
    "classes": FTTH_CLASSES,
    "class_to_id": CLASS_TO_ID,
}
with open(f"{OUTPUT_DIR}/config.json", "w") as f:
    json.dump(config, f, indent=2)

print("✅ Fine-tuning completado!")

In [ ]:
# 12. Evaluar en test set
import os
from collections import Counter

TEST_PATH = "/content/FTTH-Copilot/dataset/laya-ftth/v3/train/test/laya-format.jsonl"

print("📊 Evaluando en test set...")

# Cargar test set
test_records = []
with open(TEST_PATH) as f:
    for line in f:
        test_records.append(json.loads(line))

# Evaluar
model.eval()
correct = 0
total = 0
class_stats = Counter()

for record in test_records:
    true_class = record["answers"].get("event_class", {}).get("choice", "UNKNOWN")
    if true_class not in CLASS_TO_ID:
        continue
    
    inputs = tokenizer(record["input"], return_tensors="pt", truncation=True, max_length=128)
    inputs = {k: v.cuda() for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        pred_idx = outputs.logits.argmax().item()
    
    pred_class = ID_TO_CLASS[pred_idx]
    
    if pred_class == true_class:
        correct += 1
    total += 1
    class_stats[true_class] += 1

print(f"\nAccuracy: {correct}/{total} = {correct/total*100:.1f}%")
print("\nPor clase:")
for cls in sorted(class_stats.keys()):
    print(f"  {cls}")

In [ ]:
# 13. Subir a HuggingFace (opcional)
# Descomentar y ejecutar si querés subir el modelo

# from huggingface_hub import HfApi, create_repo
# 
# api = HfApi()
# repo_url = create_repo("tu-usuario/ftth-laya-v1", exist_ok=True)
# api.upload_folder(
#     folder_path=OUTPUT_DIR,
#     repo_id=f"tu-usuario/ftth-laya-v1",
#     repo_type="model",
# )
# print(f"Modelo subido: https://huggingface.co/tu-usuario/ftth-laya-v1")